# Customer Churn Prediction

This notebook demonstrates how to use the trained Artificial Neural Network (ANN) model to predict whether a customer is likely to churn.

The prediction pipeline includes:

- Loading the trained model
- Loading preprocessing objects
- Encoding categorical features
- Feature scaling
- Making predictions
- Displaying churn probability
- Interpreting the prediction

In [1]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

In [2]:
### Load the trained model, scaler pickle,onehot
model=load_model('customer_churn_final_optimized.keras')

## load the encoder and scaler
with open('geography_ohe.pkl','rb') as file:
    label_encoder_geo=pickle.load(file)

with open('label_encoder_gender.pkl', 'rb') as file:
    label_encoder_gender = pickle.load(file)

with open('scaler.pkl', 'rb') as file:
    scaler = pickle.load(file)

2026-08-01 14:33:45.485098: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2026-08-01 14:33:45.485117: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 24.00 GB
2026-08-01 14:33:45.485121: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 8.88 GB
I0000 00:00:1785575025.485131   31590 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1785575025.485144   31590 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [3]:
def predict_customer(input_data):

    # -----------------------------
    # Convert dictionary to DataFrame
    # -----------------------------
    input_df = pd.DataFrame([input_data])

    print("=" * 60)
    print("Customer Information")
    print("=" * 60)

    display(input_df)

    # -----------------------------
    # One-Hot Encode Geography
    # -----------------------------
    geo_encoded = label_encoder_geo.transform(
        [[input_data["Geography"]]]
    )

    geo_encoded_df = pd.DataFrame(
        geo_encoded,
        columns=label_encoder_geo.get_feature_names_out(["Geography"])
    )

    # -----------------------------
    # Label Encode Gender
    # -----------------------------
    input_df["Gender"] = label_encoder_gender.transform(
        input_df["Gender"]
    )

    # -----------------------------
    # Combine Features
    # -----------------------------
    input_df = pd.concat(
        [
            input_df.drop("Geography", axis=1),
            geo_encoded_df
        ],
        axis=1
    )

    print("\nProcessed Features")

    display(input_df)

    # -----------------------------
    # Feature Scaling
    # -----------------------------
    input_scaled = scaler.transform(input_df)

    # -----------------------------
    # Prediction
    # -----------------------------
    prediction = model.predict(
        input_scaled,
        verbose=0
    )

    probability = float(prediction[0][0])

    prediction_class = (
        "Churn"
        if probability >= 0.5
        else "No Churn"
    )

    print("\n" + "=" * 60)
    print("Prediction Result")
    print("=" * 60)

    print(f"Prediction            : {prediction_class}")
    print(f"Churn Probability     : {probability:.2%}")

    result = pd.DataFrame({

        "Prediction": [prediction_class],

        "Probability": [round(probability, 4)]

    })

    return result

In [4]:
customer = {

    "CreditScore": 600,
    "Geography": "France",
    "Gender": "Male",
    "Age": 40,
    "Tenure": 3,
    "Balance": 60000,
    "NumOfProducts": 2,
    "HasCrCard": 1,
    "IsActiveMember": 1,
    "EstimatedSalary": 50000

}

result = predict_customer(customer)

display(result)

Customer Information


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000



Processed Features


/Users/milind/Downloads/ANN_Classification/venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,0.0,0.0



Prediction Result
Prediction            : No Churn
Churn Probability     : 4.71%


2026-08-01 14:33:45.739476: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


,Prediction,Probability
0,No Churn,0.0471


In [5]:
customer2 = {

    "CreditScore": 820,
    "Geography": "Germany",
    "Gender": "Female",
    "Age": 55,
    "Tenure": 8,
    "Balance": 150000,
    "NumOfProducts": 1,
    "HasCrCard": 1,
    "IsActiveMember": 0,
    "EstimatedSalary": 90000

}

predict_customer(customer2)

Customer Information


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,820,Germany,Female,55,8,150000,1,1,0,90000



Processed Features


/Users/milind/Downloads/ANN_Classification/venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_Germany,Geography_Spain
0,820,0,55,8,150000,1,1,0,90000,1.0,0.0



Prediction Result
Prediction            : Churn
Churn Probability     : 86.61%


,Prediction,Probability
0,Churn,0.8661
